In [ ]:
!pip install Augmentor grad_cam

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models.vision_transformer import vit_b_16
from torch.cuda.amp import autocast, GradScaler
import numpy as np
from PIL import Image
import os, shutil, random, time, warnings
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import Augmentor
import wandb
from pytorch_grad_cam import GradCAM
warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Constants & Model Config

In [ ]:
labels = ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash',
          'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
num_classes = len(labels)

RAW_PATH = '/kaggle/input/datasets/djihadelhakbelkhir/realwaste/RealWaste/'
PROCESSED_BASE = '/kaggle/working/RealWaste_processed'
TRAIN_PATH = os.path.join(PROCESSED_BASE, 'train')
VAL_PATH   = os.path.join(PROCESSED_BASE, 'val')
TEST_PATH  = os.path.join(PROCESSED_BASE, 'test')

MODEL_CONFIG = {
    'vit':       {'size': 224, 'freeze_points': [10, 5, 0]},
    'densenet':  {'size': 224, 'freeze_points': [143, 80, 0]},
    'inception': {'size': 299, 'freeze_points': [249, 120, 0]},
    'mobilenet': {'size': 224, 'freeze_points': [149, 70, 0]},
    'vgg':       {'size': 224, 'freeze_points': [11, 5, 0]},
    'resnet':    {'size': 224, 'freeze_points': [140, 80, 0]},
}

BATCH_SIZE_BASE = 32
EPOCHS_PER_STAGE = 5

## Preprocessing + Augmentation 

In [ ]:
def preprocess_and_augment():
    for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
        os.makedirs(p, exist_ok=True)
        for lbl in labels:
            os.makedirs(os.path.join(p, lbl), exist_ok=True)

    print("=== 1. Splitting 80/10/10 ===")
    for label in tqdm(labels):
        src = os.path.join(RAW_PATH, label)
        files = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.png','.jpeg'))]
        random.shuffle(files)
        n = len(files)
        train_n = int(0.8 * n)
        val_n   = int(0.1 * n)
        for i, f in enumerate(files):
            src_file = os.path.join(src, f)
            if i < train_n:
                dst = os.path.join(TRAIN_PATH, label, f"train_{label}_{i}.jpg")
            elif i < train_n + val_n:
                dst = os.path.join(VAL_PATH, label, f"val_{label}_{i}.jpg")
            else:
                dst = os.path.join(TEST_PATH, label, f"test_{label}_{i}.jpg")
            shutil.copy(src_file, dst)

    # RGB fix
    def fix_rgb(folder):
        for label in labels:
            path = os.path.join(folder, label)
            for f in tqdm(os.listdir(path), desc=f"RGB {label}"):
                with Image.open(os.path.join(path, f)) as im:
                    if im.mode != 'RGB':
                        im = im.convert('RGB')
                    im.save(os.path.join(path, f))

    print("=== 2. RGB Fix ===")
    fix_rgb(TRAIN_PATH); fix_rgb(VAL_PATH); fix_rgb(TEST_PATH)

    # Geometric augmentation with Augmentor (exactly what you asked for)
    print("=== 3. Geometric Augmentation (Augmentor) ===")
    def augment_folder(folder, label, multiplier=3):
        p = Augmentor.Pipeline(os.path.join(folder, label))
        p.flip_left_right(probability=1.0)
        p.random_distortion(probability=1.0, grid_width=5, grid_height=5, magnitude=4)
        p.rotate(probability=1.0, max_left_rotation=25, max_right_rotation=25)
        p.shear(probability=1.0, max_shear_left=15, max_shear_right=15)
        p.sample(multiplier * len(os.listdir(os.path.join(folder, label))))

    for label in labels:
        augment_folder(TRAIN_PATH, label, multiplier=3)
        augment_folder(VAL_PATH,   label, multiplier=2)

    print("✅ Preprocessing + Augmentation complete!")

preprocess_and_augment()

## DataLoaders (model-specific size)

In [ ]:
def get_dataloaders(model_name):
    size = MODEL_CONFIG[model_name]['size']
    bs = 20 if size == 299 else BATCH_SIZE_BASE

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.3, 0.3, 0.3, 0.15),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.3),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize(320 if size==299 else 256),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])

    train_ds = datasets.ImageFolder(TRAIN_PATH, train_tf)
    val_ds   = datasets.ImageFolder(VAL_PATH,   val_tf)
    test_ds  = datasets.ImageFolder(TEST_PATH,  val_tf)

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)

    # === CRITICAL FIX: force float32 weights ===
    counts = np.zeros(num_classes, dtype=np.int64)
    for _, lbl in train_ds:
        counts[lbl] += 1
    class_weights = torch.tensor(1.0 / (counts + 1e-6), dtype=torch.float32).to(DEVICE)

    print(f"✅ {model_name} loaders ready | batch={bs} | weights=float32")
    return train_loader, val_loader, test_loader, class_weights